## Upgrade your model to Wflow.jl version 1

### Upgrading a Wflow SBM model

We can call the `upgrade_to_v1_wflow` function using the `update` CLI of HydroMT. Here is the configuration file that we will use:

In [2]:
fn_config = "wflow_update_v1_sbm.yml"
with open(fn_config, "r") as f:
    txt = f.read()
print(txt)

global:
  config_filename: wflow_sbm_v0x.toml

steps:
  - upgrade_to_v1_wflow:

  - staticmaps.write:
      filename: staticmaps_v1.nc

  - config.write:
      filename: wflow_sbm_v1.toml

  # If you have any lake_*.csv tables, write them too.
  - tables.write:



In this config, we start by setting the name of the TOML file of the version 0x model (if different than *wflow_sbm.toml*), call the `upgrade_to_v1_wflow` method, and then save the updated TOML file into a new *wflow_sbm_v1.toml* file.

Note that if you use the standard *wflow_sbm.toml* name and do not mind it being overwritten, the above example can be reduced into only calling `upgrade_to_v1_wflow`.

Let's now call HydroMT update to update our old config (in this case we do not need a data catalog):

In [3]:
!hydromt update wflow_sbm "./data/wflow_upgrade/sbm" -i wflow_update_v1_sbm.yml -v

2026-03-09 15:47:04,128 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 15:47:04,192 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-03-09 15:47:04,192 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 15:47:04,241 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 15:47:04,241 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from C:/Data/TUD/MSc_CE/Courses/2nd_year/7.CIE5060_Thesis/Codes/MSc_thesis_hydromt_wflow/examples/data/wflow_upgrade/sbm/wflow_sbm_v0x.toml.
2026-03-09 15:47:04,243 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 15:47:04,245 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 15:47:06,558 - hydromt.model.model - model - INFO - update: upgrade_to_v1_wfl

And let's see the results. Here is what our old TOML file looked like:

In [4]:
toml_v0x = "./data/wflow_upgrade/sbm/wflow_sbm_v0x.toml"
with open(toml_v0x, "r") as f:
    txt = f.read()
print(txt)

casename = "wflow_sbm"
calendar = "proleptic_gregorian"
starttime = "2010-02-02T00:00:00"
endtime = "2010-02-10T00:00:00"
time_units = "days since 1900-01-01 00:00:00"
timestepsecs = 86400
dir_output = "run_default"


[state]
path_input = "instate/instates.nc"
path_output = "outstate/outstates.nc"

[input]
path_forcing = "inmaps.nc"
path_static = "staticmaps_v0x.nc"
gauges = "wflow_gauges"
ldd = "wflow_ldd"
river_location = "wflow_river"
subcatchment = "wflow_subcatch"
forcing = [
    "vertical.precipitation",
    "vertical.temperature",
    "vertical.potential_evaporation",
]
cyclic = ["vertical.leaf_area_index"]
gauges_grdc = "wflow_gauges_grdc"
reservoirlocs = "wflow_reservoirlocs"

[model]
type = "sbm"
masswasting = true
snow = true
reinit = true
reservoirs = true
lakes = true
glacier = true
land_routing = "kinematic-wave"
river_routing = "kinematic-wave"
kin_wave_iteration = true
kw_river_tstep = 900
kw_land_tstep = 3600
thicknesslayers = [100, 300, 800]

[state.vertical]
satwater

And here is the same TOML in version 1 format:

In [5]:
toml_v1 = "./data/wflow_upgrade/sbm/wflow_sbm_v1.toml"
with open(toml_v1, "r") as f:
    txt = f.read()
print(txt)

dir_output = "run_default"

[time]
calendar = "proleptic_gregorian"
starttime = "2010-02-02T00:00:00"
endtime = "2010-02-10T00:00:00"
time_units = "days since 1900-01-01 00:00:00"
timestepsecs = 86400

[model]
type = "sbm"
snow_gravitational_transport__flag = true
snow__flag = true
cold_start__flag = true
reservoir__flag = true
glacier__flag = true
land_routing = "kinematic_wave"
river_routing = "kinematic_wave"
kinematic_wave__adaptive_time_step_flag = true
river_kinematic_wave__time_step = 900
land_kinematic_wave__time_step = 3600
soil_layer__thickness = [
    100,
    300,
    800,
]

[state]
path_input = "instate/instates.nc"
path_output = "outstate/outstates.nc"

[state.variables]
vegetation_canopy_water__depth = "canopystorage"
soil_water_saturated_zone__depth = "satwaterdepth"
soil_layer_water_unsaturated_zone__depth = "ustorelayerdepth"
soil_surface__temperature = "tsoil"
snowpack_dry_snow__leq_depth = "snow"
snowpack_liquid_water__depth = "snowwater"
glacier_ice__leq_depth = "

The lakes and reservoirs have been merged into one structure and the new staticmaps file has been generated. Here are all the available variables:

In [6]:
import xarray as xr

staticmaps = xr.open_dataset("./data/wflow_upgrade/sbm/staticmaps_v1.nc")
print(list(staticmaps.data_vars.keys()))

['x_out', 'y_out', 'idx_out', 'wflow_ldd', 'wflow_subcatch', 'wflow_uparea', 'subare', 'dem_subgrid', 'wflow_streamorder', 'wflow_dem', 'Slope', 'wflow_river', 'wflow_riverlength', 'RiverSlope', 'N_River', 'wflow_riverwidth', 'RiverDepth', 'LakeAvgOut', 'wflow_glacierareas', 'wflow_glacierstore', 'wflow_glacierfrac', 'wflow_landuse', 'Kext', 'N', 'PathFrac', 'RootingDepth', 'Sl', 'Swood', 'WaterFrac', 'kc', 'alpha_h1', 'h1', 'h2', 'h3_high', 'h3_low', 'h4', 'LAI', 'thetaS', 'thetaR', 'SoilThickness', 'SoilMinThickness', 'c', 'KsatVer', 'KsatVer_0.0cm', 'KsatVer_5.0cm', 'KsatVer_15.0cm', 'KsatVer_30.0cm', 'KsatVer_60.0cm', 'KsatVer_100.0cm', 'KsatVer_200.0cm', 'M_original_', 'M_', 'f_', 'M_original', 'M', 'f', 'wflow_soil', 'wflow_gauges', 'wflow_gauges_grdc', 'KsatHorFrac', 'Cfmax', 'cf_soil', 'EoverR', 'InfiltCapPath', 'InfiltCapSoil', 'MaxLeakage', 'rootdistpar', 'TT', 'TTI', 'TTM', 'WHC', 'G_Cfmax', 'G_SIfrac', 'G_TT', 'reservoir_b', 'reservoir_e', 'reservoir_outflow_threshold', 're

### Upgrading a Wflow Sediment model

We can call the `upgrade_to_v1_wflow` function using the `update` CLI of HydroMT. Here is the configuration file that we will use:

In [7]:
fn_config = "wflow_update_v1_sediment.yml"
with open(fn_config, "r") as f:
    txt = f.read()
print(txt)

global:
  config_filename: wflow_sediment_v0x.toml

steps:
  - upgrade_to_v1_wflow:
      soil_fn: soilgrids

  - staticmaps.write:
      filename: staticmaps_v1.nc

  - config.write:
      filename: wflow_sediment_v1.toml



In this config, we start by setting the name of the TOML file of the version 0x model (if different than *wflow_sediment.toml*), call the `upgrade_to_v1_wflow` method, and then save the updated TOML file into a new *wflow_sbm_v1.toml* file and the updated staticmaps file with the additional parameters in *staticmaps_v1.toml*.

Note that if you use the standard *wflow_sediment.toml* name and do not mind it or the staticmaps file being overwritten, the above example can be reduced into only calling `upgrade_to_v1_wflow`.

Let's now call HydroMT update to update our old config (in this case we do need a data catalog as we will update some of the soil parameters):

In [8]:
!hydromt update wflow_sediment "./data/wflow_upgrade/sediment" -i wflow_update_v1_sediment.yml -d artifact_data -v

2026-03-09 15:47:14,226 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 15:47:14,637 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Reading data catalog artifact_data latest
2026-03-09 15:47:14,637 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\.hydromt\artifact_data\v1.0.0\data_catalog.yml
2026-03-09 15:47:15,399 - hydromt.model.model - model - INFO - Initializing wflow_sediment model from hydromt_wflow (v1.0.1).
2026-03-09 15:47:15,399 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 15:47:15,419 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 15:47:15,419 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from C:/Data/TUD/MSc_CE/Courses/2nd_year/7.CIE5060_Thesis/Codes/MSc_th

And let's see the results. Here is what our old TOML file looked like:

In [9]:
toml_v0x = "./data/wflow_upgrade/sediment/wflow_sediment_v0x.toml"
with open(toml_v0x, "r") as f:
    txt = f.read()
print(txt)

casename = "wflow_sediment"
calendar = "proleptic_gregorian"
starttime = "2010-02-01T00:00:00"
endtime = "2010-02-10T00:00:00"
time_units = "days since 1900-01-01 00:00:00"
timestepsecs = 86400
dir_output = "run_default"

[state]
path_input = "instate/instates.nc"
path_output = "outstate/outstates.nc"

[input]
path_forcing = "inmaps.nc"
path_static = "staticmaps_v0x.nc"
ldd = "wflow_ldd"
river_location = "wflow_river"
subcatchment = "wflow_subcatch"
forcing = [ "vertical.h_land", "vertical.interception", "vertical.precipitation", "vertical.q_land", "lateral.river.h_riv", "lateral.river.q_riv",]
cyclic = [ "vertical.leaf_area_index",]
gauges = "wflow_gauges"
gauges_grdc = "wflow_gauges_grdc"

[model]
type = "sediment"
reinit = true
runrivermodel = true
doreservoir = true
dolake = true
rainerosmethod = "answers"
landtransportmethod = "yalinpart"
rivtransportmethod = "bagnold"

[output]
path = "output.nc"

[csv]
path = "output.csv"
[[csv.column]]
header = "TSS"
map = "gauges"
parameter = 

And here is the same TOML in version 1 format:

In [10]:
toml_v1 = "./data/wflow_upgrade/sediment/wflow_sediment_v1.toml"
with open(toml_v1, "r") as f:
    txt = f.read()
print(txt)

dir_output = "run_default"

[time]
calendar = "proleptic_gregorian"
starttime = "2010-02-01T00:00:00"
endtime = "2010-02-10T00:00:00"
time_units = "days since 1900-01-01 00:00:00"
timestepsecs = 86400

[model]
type = "sediment"
cold_start__flag = true
run_river_model__flag = true
reservoir__flag = true
rainfall_erosion = "answers"
land_transport = "yalinpart"
river_transport = "bagnold"

[state]
path_input = "instate/instates.nc"
path_output = "outstate/outstates.nc"

[state.variables]
river_water_clay__mass = "clayload"
river_bed_clay__mass = "claystore"
river_water_clay__mass_flow_rate = "outclay"
river_water_gravel__mass = "gravload"
river_bed_gravel__mass = "gravstore"
river_water_gravel__mass_flow_rate = "outgrav"
river_water_large_aggregates__mass = "laggload"
river_bed_large_aggregates__mass = "laggstore"
river_water_large_aggregates__mass_flow_rate = "outlagg"
river_water_small_aggregates__mass = "saggload"
river_bed_small_aggregates__mass = "saggstore"
river_water_small_aggreg

You can also note that the new staticmaps file was generated. Here are all the available variables:

In [11]:
import xarray as xr

staticmaps = xr.open_dataset("./data/wflow_upgrade/sediment/staticmaps_v1.nc")
print(list(staticmaps.data_vars.keys()))

['x_out', 'y_out', 'idx_out', 'wflow_ldd', 'wflow_subcatch', 'wflow_uparea', 'subare', 'dem_subgrid', 'wflow_streamorder', 'wflow_dem', 'Slope', 'wflow_river', 'wflow_riverlength', 'RiverSlope', 'N_River', 'wflow_riverwidth', 'RiverDepth', 'ResMaxVolume', 'ResDemand', 'ResMaxRelease', 'ResTargetFullFrac', 'ResTargetMinFrac', 'LakeAvgLevel', 'LakeAvgOut', 'Lake_b', 'Lake_e', 'LakeStorFunc', 'LakeOutflowFunc', 'LakeThreshold', 'LinkedLakeLocs', 'D50_River', 'ClayF_River', 'SiltF_River', 'SandF_River', 'GravelF_River', 'wflow_landuse', 'Kext', 'PathFrac', 'Sl', 'Swood', 'USLE_C', 'LAI', 'CanopyHeight', 'PercentClay', 'PercentSilt', 'PercentOC', 'ErosK', 'USLE_K', 'wflow_gauges', 'wflow_gauges_grdc', 'c_Bagnold', 'eros_expo', 'eros_ov', 'eros_spl_EUROSEM', 'exp_Bagnold', 'erosion_soil_detachability', 'erosion_usle_k', 'soil_sediment_d50', 'land_govers_c', 'land_govers_n', 'soil_clay_fraction', 'soil_silt_fraction', 'soil_sagg_fraction', 'soil_sand_fraction', 'soil_lagg_fraction', 'river_be